**Fig. 6 and 7, step 5 (notebook 05): compare the hair placode with the ridge basal cells (Fig. 7G, 7I-L)**

In [ ]:
# Working directory: data/fig6_7/ under the repository root, where the Fig. 6 and 7 deposit folder is downloaded
# (all_stacks_features.csv, workspace/ = the plugin's Step 4 workspace with the 14 stacks (13 analysed) and their meshes,
# run/ = what these notebooks write and read back, figures/ = the saved panels).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig6_7")
print("working directory:", os.getcwd())

# 05 — Hair placode vs fingerprint ridge (within one donor)

**Question.**  Within ONE donor at ONE stage (donor2, 13 weeks) — so donor and
stage are controlled — do hair placode cells and ridge basal cells differ in
shape *beyond* the difference between their surrounding tissues
(hair-bearing front skin vs volar digit skin)?

Four groups, in this order: `placode_background`, `hair_placode`, `ridge`,
`ridge_background`.  The core result is how far each structure departs from
**its own** surrounding epidermis, and whether the two departures look alike
or differ.

Data: the final 31-feature run (`morphospace_results.pkl`,
`niche_results.pkl`) — the run of Fig. 7.  Notebooks 01 / 02 / 04 and the
pickles are not modified.  Helper functions are copied verbatim from notebook
04 so the plots match the figure notebooks.

## Statistics

* Sections 1-5 report effect sizes with bootstrap 95% CIs; section 6 adds the four-group tests of the Fig. 7 legend, treating cells as replicates, so its p-values describe this donor only.
* Bootstrap CIs resample cells within each group, so they reflect sampling
  of cells, NOT donor-to-donor variation — they are narrower than a
  between-donor interval would be.
* Structure cells were **selected by niche clusters**, and the niche
  embedding is built from the same shape PCs (each cell's niche = its
  neighbours' shape).  Some shape separation of the structures from their
  background is therefore expected **by construction**.  The informative part
  is the comparison *relative to the backgrounds*: placode vs ridge against
  background vs background, and the two departure profiles against each
  other.
* Cliff's delta (δ) = P(a > b) − P(a < b), from −1 to 1; 0 = no shift.  Rough
  guide (Romano et al. 2006): |δ| < 0.147 negligible, < 0.33 small, < 0.474
  medium, ≥ 0.474 large.  δ depends only on ranks, so a log transform does
  not change it (LOG_FEATURES only affects the plots).
* **Orientation.**  `z_span`, `z_elongation`, `xy_elongation` and `aspect_ratio`
  are measured along the microscope axes, so they mix how elongated a cell is
  with how it is tilted.  `major_axis_z_angle` is the tilt itself (angle of
  the cell's long axis to z: 0 = upright, 90 = lying flat); `inertia_elongation`,
  `axis_minor_length` and `axis_major_length` describe the cell's own shape
  whatever its orientation.  The samples are flat, so z is close to the skin's
  apical-basal axis.
* `roughness` is left out of FEATURES: it is exactly 1 / `true_sphericity`
  and would mirror it with the sign flipped.  It stays inside the PCs.
* The last section (the four-group statistics cells) reports the four-group tests described in the Fig. 7 legend (Welch's ANOVA with Games-Howell post hoc).

## Config — edit here

In [ ]:
# ---- data (the final 31-feature run = Fig. 7) ----
MORPHO_PKL = r"run/morphospace_results.pkl"
NICHE_PKL  = r"run/niche_results.pkl"
NICHE_CONFIG      = "hop3_pure"          # Fig. 6G, 7F-G
NICHE_RESOLUTION  = 1.5                  # Fig. 6G, 7F-G (20 clusters)
NICHE_COL         = f"niche_leiden_r{NICHE_RESOLUTION}"   # the niche cluster IDs below are at resolution 1.5
SHAPE_CLUSTER_COL = "leiden"

# ---- the comparison ----
RIDGE_STACK   = "(13 wks) D3L ventral ridges x40-Smooth05_Resampled.tif"
PLACODE_STACK = "(13 wks) Front skin x40-Smooth05_Resampled.tif"
RIDGE_CLUSTERS              = [3, 10, 13, 14, 7]   # ridge basal cells: 3 = ridge core; 10, 13, 14, 7 = further basal cells
RIDGE_BACKGROUND_CLUSTERS   = None   # None = all other cells in RIDGE_STACK; else list of basal interridge clusters
PLACODE_CLUSTERS            = [17]   # hair placode basal cells
PLACODE_BACKGROUND_CLUSTERS = [0]    # surrounding epidermis; None = all other cells in PLACODE_STACK

FEATURES = ["PC1", "PC2", "PC3", "z_span", "area", "true_sphericity", "aspect_ratio",
            "z_elongation", "xy_elongation", "apical_basal_area_ratio",
            "cross_section_taper", "solidity",
            # orientation vs intrinsic shape (z-based features mix the two):
            "major_axis_z_angle",    # tilt of the long axis from z: 0 = upright, 90 = lying flat
            "inertia_elongation",    # elongation independent of orientation
            "axis_minor_length",     # cell width (orientation-free)
            "axis_major_length"]     # cell length (orientation-free)
LOG_FEATURES = ["area"]      # log1p for plots (effect sizes are rank-based)
N_PCS = 10                   # shape PC space for the multivariate distances

N_BOOT = 2000
SEED = 0
N_REPRESENTATIVES = 10

# ---- output (under data/fig6_7, git-ignored) ----
OUT_TABLES = "run/placode_vs_ridge"
FIG_PREFIX = "Fig7_placode_ridge_"

## Helpers (copied verbatim from notebook 04)

In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go

FONT_FAMILY       = "Helvetica Neue, Arial"
FONT_AXIS_TITLE   = 13
FONT_TICK         = 11
FONT_LEGEND       = 10.5
FONT_LEGEND_TITLE = 11
_CLUSTER_BASE = [
    "#0077BB", "#EE7733", "#009988", "#CC3311", "#EE3377",
    "#33BBEE", "#AA3377", "#BB5566", "#44AA99", "#DDCC77",
    "#882255", "#117733", "#88CCEE", "#332288", "#CC6677",
    "#EE9944", "#99DDBB", "#AA4499", "#661100", "#6699CC",
    "#888888", "#D4B483",
]
DONOR_PALETTE = {
    "donor1": "#332288",
    "donor2": "#CC6677",
    "donor3": "#44AA99",
    "donor4": "#88CCEE",
    "donor5": "#DDCC77",
}
MM_PER_INCH = 25.4
SVG_DPI = 72
PNG_DPI = 300
FIG_OUT_DIR = 'figures'
_CAM_CFGS = {
    "iso": dict(pos=(2.8, 2.2, 1.6), up=(0, 0, 1)),
    "xz":  dict(pos=(0.0, -3.2, 0.0), up=(0, 0, 1)),
    "xy":  dict(pos=(0.0, 0.0, 3.2), up=(0, 1, 0)),
    "yz":  dict(pos=(3.2, 0.0, 0.0), up=(0, 0, 1)),
}
SEED_HIGHLIGHT = 0


def _hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


def _fmt_cluster(v):
    """Format a cluster label: integer-valued floats become int strings.

    Niche cluster columns are stored as float so they can hold NaN for
    cells outside the valid mask. Without this helper, str(1.0) shows as
    "1.0" which looks wrong in tick labels.
    """
    if v is None:
        return "NA"
    try:
        import math as _math
        f = float(v)
        if _math.isnan(f):
            return "NA"
        if f.is_integer():
            return str(int(f))
    except (TypeError, ValueError):
        pass
    return str(v)


def apply_transform(values, method="raw", winsorize_sigma=5.0):
    """
    Feature transform matching the EpiCycle GUI (see
    napari_epicycle/segmentation/feature_stats.py:179-226).
    Preserves NaN.

    method : 'raw' | 'log1p' | 'sqrt' | 'rank' | 'zscore' | 'winsorize'

    Notes
    -----
    - log1p / sqrt shift by min if any finite value is negative so the
      transform is always defined.
    - rank normalizes to (0, 1] via scipy.stats.rankdata(x) / n.
    - winsorize clips to +/- winsorize_sigma std-devs then unscales.
    """
    arr = np.asarray(values, dtype=np.float64).copy()
    finite_mask = np.isfinite(arr)
    if not finite_mask.any() or method == "raw":
        return arr
    finite = arr[finite_mask]

    if method in ("log", "log1p"):
        if finite.min() < 0:
            arr[finite_mask] = arr[finite_mask] - finite.min()
            finite = arr[finite_mask]
        arr[finite_mask] = np.log1p(finite)
        return arr

    if method == "sqrt":
        if finite.min() < 0:
            arr[finite_mask] = arr[finite_mask] - finite.min()
            finite = arr[finite_mask]
        arr[finite_mask] = np.sqrt(finite)
        return arr

    if method == "rank":
        from scipy.stats import rankdata
        arr[finite_mask] = rankdata(finite) / len(finite)
        return arr

    if method == "zscore":
        if len(finite) > 1:
            mu = np.mean(finite)
            std = np.std(finite, ddof=1)
            if std > 0:
                arr[finite_mask] = (finite - mu) / std
        return arr

    if method == "winsorize":
        if len(finite) > 1:
            mu = np.mean(finite)
            std = np.std(finite, ddof=1)
            if std > 0:
                z = (finite - mu) / std
                z = np.clip(z, -winsorize_sigma, winsorize_sigma)
                arr[finite_mask] = z * std + mu
        return arr

    raise ValueError(f"Unknown transform method: {method!r}")


def _base_layout(width, height, legend_title=None, show_legend=True):
    return dict(
        plot_bgcolor="#fff", paper_bgcolor="#fff",
        width=width, height=height,
        margin=dict(l=65, r=180, t=30, b=60),  # r=180 so external legends don't clip in SVG
        font=dict(family=FONT_FAMILY),
        showlegend=show_legend,
        legend=dict(
            font=dict(size=FONT_LEGEND, family=FONT_FAMILY),
            title=dict(
                text=legend_title or "",
                font=dict(size=FONT_LEGEND_TITLE, family=FONT_FAMILY),
            ),
            tracegroupgap=4,
            x=1.01, y=0.98, xanchor="left",
            bgcolor="rgba(255,255,255,0.9)",
            itemsizing="constant",
        ),
    )


def _apply_umap_axes(fig, x_title="UMAP 1", y_title="UMAP 2"):
    axkw = dict(
        showgrid=True, gridcolor="rgba(200,200,200,0.25)", gridwidth=0.5,
        zeroline=False,
        showticklabels=False, ticks="",
        showline=False,
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
    )
    fig.update_xaxes(title_text=x_title, **axkw)
    fig.update_yaxes(title_text=y_title, **axkw)


def violin_panel(df, feature, group_col, show_legend=True,
                 palette=None, group_order=None,
                 show_box=True, show_points=True, max_points=200,
                 show_median_line=False, baseline=None,
                 violin_width=0.6, box_width=0.3, jitter_frac=0.12,
                 violin_fill_alpha=0.08, violin_line_alpha=0.4,
                 point_size=3.5, point_opacity=0.32,
                 y_transform=None,
                 width=760, height=500,
                 y_title=None, x_title=None,
                 tick_rotation=0,
                 show_n_annotation=True,
                 include_zero=True,
                 random_seed=42):
    """
    Single-feature violin across the categories in `group_col`.
    Aesthetic: half-violin + overlaid narrow box + jittered scatter (side-offset),
    optional median trend line + baseline reference.

    Parameters
    ----------
    df : DataFrame
    feature : str  — numeric column to plot on y
    group_col : str  — categorical column; one violin per value
    palette : dict {group_value: hex_color} or None (auto)
    group_order : list of group values in display order (default: sorted)
    show_box / show_points : toggle overlay elements
    max_points : subsample threshold for the jitter scatter
    show_median_line : connect group medians (dashed red) — useful for
                      ordered groups like timepoint, off for Leiden clusters
    baseline : optional horizontal reference line (y-value)
    violin_width / box_width / jitter_frac : geometry controls
    violin_fill_alpha / violin_line_alpha / point_opacity : visual weights
    y_transform : None | 'raw' | 'log1p' | 'sqrt' | 'rank' | 'zscore' | 'winsorize'
        Matches the EpiCycle GUI transform set (feature_stats.apply_transform).
    tick_rotation : degrees on x labels (useful for sample names)
    """
    import numpy as _np
    d = df[[feature, group_col]].dropna().copy()

    if y_transform and y_transform != "raw":
        d[feature] = apply_transform(d[feature].values, method=y_transform)

    if group_order is None:
        try:
            group_order = sorted(d[group_col].unique(),
                                 key=lambda x: float(x))
        except (TypeError, ValueError):
            group_order = sorted(d[group_col].unique(), key=str)

    if palette is None:
        palette = {g: _CLUSTER_BASE[i % len(_CLUSTER_BASE)]
                   for i, g in enumerate(group_order)}

    rng = _np.random.default_rng(random_seed)
    fig = go.Figure()
    legend_added_median = False

    for i, g in enumerate(group_order):
        sub = d[d[group_col] == g][feature]
        if sub.empty:
            continue
        c_hex = palette.get(g, "#888888")
        r, gr, b = _hex_to_rgb(c_hex)
        fill_rgba = f"rgba({r},{gr},{b},{violin_fill_alpha:.3f})"
        line_rgba = f"rgba({r},{gr},{b},{violin_line_alpha:.3f})"

        # Half-violin (positive side)
        fig.add_trace(go.Violin(
            y=sub, x0=i,
            side="positive", width=violin_width,
            line=dict(color=line_rgba, width=1),
            fillcolor=fill_rgba,
            meanline_visible=False,
            showlegend=False, hoverinfo="skip", points=False,
            scalemode="width",
        ))

        if show_box:
            fig.add_trace(go.Box(
                y=sub, x0=i, orientation="v",
                boxpoints=False, width=box_width, whiskerwidth=0.4,
                line=dict(color="#666666", width=1.0),
                fillcolor="rgba(100,100,100,0.04)",
                showlegend=False, hoverinfo="skip",
                marker=dict(opacity=0),
            ))

        if show_points:
            pts = sub.sample(max_points, random_state=random_seed) \
                if len(sub) > max_points else sub
            jitter = rng.uniform(-jitter_frac, jitter_frac, size=len(pts))
            fig.add_trace(go.Scatter(
                x=i + jitter - 0.15,
                y=pts.values, mode="markers",
                marker=dict(size=point_size, color=c_hex,
                            opacity=point_opacity, line=dict(width=0)),
                showlegend=False,
                hovertemplate=(
                    f"<b>{g}</b><br>"
                    f"{feature}: %{{y:.3f}}<extra></extra>"
                ),
            ))

    if show_median_line:
        medians = [d[d[group_col] == g][feature].median() for g in group_order]
        fig.add_trace(go.Scatter(
            x=list(range(len(group_order))), y=medians,
            mode="lines+markers",
            line=dict(color="#CC3311", width=1.5, dash="dash"),
            marker=dict(symbol="circle-open", size=7, color="#CC3311",
                        line=dict(width=1.5, color="#CC3311")),
            showlegend=True, name="Median",
            hovertemplate="Group: %{x}<br>Median: %{y:.3f}<extra></extra>",
        ))
        legend_added_median = True

    if baseline is not None:
        fig.add_shape(
            type="line",
            x0=-0.5, x1=len(group_order) - 0.5,
            y0=baseline, y1=baseline,
            line=dict(color="#999", width=1, dash="dashdot"),
        )

    fig.update_layout(
        plot_bgcolor="#fff", paper_bgcolor="#fff",
        width=width, height=height,
        margin=dict(l=70, r=20, t=30, b=80 if tick_rotation else 60),
        font=dict(family=FONT_FAMILY),
        showlegend=legend_added_median,
        legend=dict(
            font=dict(size=FONT_LEGEND, family=FONT_FAMILY),
            x=1.01, y=0.98, xanchor="left",
            bgcolor="rgba(255,255,255,0.9)",
        ),
    )
    fig.update_xaxes(
        tickvals=list(range(len(group_order))),
        ticktext=[_fmt_cluster(g) for g in group_order],
        title_text=x_title or group_col,
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
        tickangle=tick_rotation,
        range=[-0.6, len(group_order) - 0.4],
        showgrid=True, gridcolor="rgba(200,200,200,0.3)", gridwidth=0.5,
        zeroline=False,
    )
    _y_suffix = "" if not y_transform or y_transform == "raw" else f" ({y_transform})"
    fig.update_yaxes(
        title_text=y_title or (feature + _y_suffix),
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
        gridcolor="rgba(200,200,200,0.3)", gridwidth=0.5,
        zeroline=False,
        rangemode="tozero" if include_zero else "normal",
    )

    if show_n_annotation:
        fig.add_annotation(
            text=f"n = {len(d):,}",
            x=0.98, y=0.03, xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=11, color="#555", family=FONT_FAMILY),
            xanchor="right", yanchor="bottom",
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="rgba(200,200,200,0.5)",
            borderwidth=1, borderpad=4,
        )
    fig.update_layout(showlegend=show_legend)
    return fig


def mm_to_px(mm, dpi=SVG_DPI):
    return int(round(mm * dpi / MM_PER_INCH))


def save_fig(fig, name, *, mm=None, width_px=None, height_px=None,
             out_dir=None, formats=("svg", "png")):
    """Save a Plotly figure as SVG (vector, Illustrator-editable) + high-dpi PNG.

    Sizing:
        mm=(w, h)            -> sized to those mm at 72 dpi (preferred).
        width_px / height_px -> raw pixel size.
        neither              -> uses fig.layout.width / height.

    PNG is rendered at PNG_DPI / SVG_DPI (~4.17x) for crisp previews.
    Requires kaleido:  pip install -U kaleido
    """
    out_dir = out_dir if out_dir is not None else FIG_OUT_DIR
    os.makedirs(out_dir, exist_ok=True)
    if mm is not None:
        width_px = mm_to_px(mm[0])
        height_px = mm_to_px(mm[1])
    png_scale = PNG_DPI / SVG_DPI

    paths = []
    for fmt in formats:
        path = os.path.join(out_dir, f"{name}.{fmt}")
        kw = {}
        if width_px:  kw["width"] = width_px
        if height_px: kw["height"] = height_px
        if fmt == "png":
            kw["scale"] = png_scale
        fig.write_image(path, format=fmt, **kw)
        paths.append(path)
        print(f"  -> {path}")
    return paths


def _axis_titles(x_col, y_col):
    """Axis titles by the notebooks' convention: niche_* columns -> 'Niche UMAP'."""
    def word(c):
        return "Niche UMAP" if c.startswith("niche_") else "UMAP"
    return f"{word(x_col)} 1", f"{word(y_col)} 2"


def umap_highlight(df, highlight_col, highlight_values,
                   x_col="umap_1", y_col="umap_2",
                   palette=None, bg_color="#D9D9D9", bg_opacity=0.25, bg_size=3,
                   fg_size=5, fg_opacity=0.85, mask=None, title=None,
                   legend_title=None, width=900, height=700,
                   on_top=None, names=None, seed=None, show_legend=True):
    """Highlight-vs-grey UMAP.  Highlights chosen groups on a UMAP (one group against
    the rest, the cells of one donor, ridged against whorled skin, small structures
    over their tissue); this notebook uses it for the four placode / ridge groups.

    Every cell is drawn first in light grey; cells whose `highlight_col` is in
    `highlight_values` (and, if given, where the boolean Series `mask` is True)
    are drawn on top in their `palette` colour, in a seeded random order so no
    group is consistently on top (notebook 03's "intermix").
    on_top : values drawn after the others (e.g. small structures over their
             tissue backgrounds); each layer is shuffled on its own.
    names  : {value: legend text}; the legend always adds the cell count.
    Works for shape (umap_1/2) and niche (niche_umap_1/2) coordinates.
    """
    seed = SEED_HIGHLIGHT if seed is None else seed
    palette, names = palette or {}, names or {}
    values = list(highlight_values)
    d = df.dropna(subset=[x_col, y_col])
    eligible = d[highlight_col].isin(values)
    if mask is not None:
        eligible &= mask.astype(bool).reindex(d.index, fill_value=False)

    fig = go.Figure()
    bg = d[~eligible]
    fig.add_trace(go.Scatter(
        x=bg[x_col], y=bg[y_col], mode="markers",
        marker=dict(size=bg_size, color=bg_color, opacity=bg_opacity,
                    line=dict(width=0)),
        name="other cells", showlegend=False, hoverinfo="skip"))

    fg = d[eligible]
    counts = fg[highlight_col].value_counts()
    top = [v for v in (on_top or []) if v in values]
    rng = np.random.default_rng(seed)
    for layer in ([v for v in values if v not in top], top):
        sub = fg[fg[highlight_col].isin(layer)]
        if sub.empty:
            continue
        sub = sub.iloc[rng.permutation(len(sub))]
        grp = sub[highlight_col]
        fig.add_trace(go.Scatter(
            x=sub[x_col].to_numpy(), y=sub[y_col].to_numpy(), mode="markers",
            marker=dict(size=fg_size, opacity=fg_opacity, line=dict(width=0),
                        color=grp.map(lambda v: palette.get(v, "#888")).to_numpy()),
            customdata=grp.map(lambda v: names.get(v, str(v))).to_numpy(),
            hovertemplate="<b>%{customdata}</b><br>%{x:.2f}, %{y:.2f}<extra></extra>",
            name="highlighted", showlegend=False))
    for rank, v in enumerate(values):          # legend-only entries, in order
        n = int(counts.get(v, 0))
        if n == 0:
            print(f"[umap_highlight]: no cells for {highlight_col} = {v!r}")
            continue
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=fg_size, color=palette.get(v, "#888"),
                        opacity=fg_opacity, line=dict(width=0)),
            name=f"{names.get(v, str(v))} (n={n:,})", legendrank=rank,
            hoverinfo="skip"))

    fig.update_layout(**_base_layout(width, height,
                                     legend_title=legend_title or f"<b>{highlight_col}</b>",
                                     show_legend=show_legend))
    _apply_umap_axes(fig)
    xt, yt = _axis_titles(x_col, y_col)
    fig.update_xaxes(title_text=xt)
    fig.update_yaxes(title_text=yt)
    if title:
        fig.update_layout(title=dict(text=title, x=0.01, xanchor="left",
                                     font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY)),
                          margin=dict(t=55))
    return fig

os.makedirs(OUT_TABLES, exist_ok=True)
print('helpers ready')

In [ ]:
def save(fig, desc):
    """save_fig (SVG + PNG) under the FIG_PREFIX placode / ridge name; if the static
    image backend (kaleido) is missing, write an interactive HTML instead."""
    name = FIG_PREFIX + desc
    try:
        return save_fig(fig, name)
    except Exception as e:
        os.makedirs(FIG_OUT_DIR, exist_ok=True)
        path = os.path.join(FIG_OUT_DIR, name + ".html")
        fig.write_html(path, include_plotlyjs="cdn")
        print(f"  [no static export: {type(e).__name__}] -> {path}")
        return [path]


def save_table(tab, name):
    path = os.path.join(OUT_TABLES, name + ".csv")
    tab.to_csv(path)
    print("  table ->", path)

## Load data (same logic as notebook 04)

In [ ]:
with open(MORPHO_PKL, "rb") as f:
    morpho = pickle.load(f)
df = morpho["df"].copy().reset_index(drop=True)
pcs = np.asarray(morpho["pcs"])
for k in range(pcs.shape[1]):
    df[f"PC{k + 1}"] = pcs[:, k]
mesh_lookup = morpho.get("mesh_lookup", {})

with open(NICHE_PKL, "rb") as f:
    niche_data = pickle.load(f)
nr = niche_data["niche_results"][NICHE_CONFIG]
valid = np.asarray(nr["valid_mask"])
col = np.full(len(df), np.nan)
col[valid] = np.asarray(nr["clusters"][NICHE_RESOLUTION]["membership"], dtype=float)
df[NICHE_COL] = col

print(f"{len(df):,} cells, {df.stack_name.nunique()} stacks, {pcs.shape[1]} PCs, "
      f"{len(morpho['avail_kf'])} features (roughness included: {'roughness' in morpho['avail_kf']})")
print(f"shape clusters ({SHAPE_CLUSTER_COL}): {df[SHAPE_CLUSTER_COL].nunique()} | "
      f"niche clusters ({NICHE_CONFIG}, r{NICHE_RESOLUTION}): {int(np.nanmax(col)) + 1}")

## Check the design and build the four groups

In [ ]:
GROUP_ORDER = ["placode_background", "hair_placode", "ridge", "ridge_background"]
GROUP_PALETTE = {                      # backgrounds = light tints of their structure
    "hair_placode":       "#AA3377",   # front-skin magenta (tissue palette)
    "placode_background": "#E3B5CF",
    "ridge":              "#CC3311",   # volar red
    "ridge_background":   "#F3B9A9",
}

meta = df.groupby("stack_name")[["donor", "timepoint", "tissue"]].first()
for s in (PLACODE_STACK, RIDGE_STACK):
    assert s in meta.index, f"stack not found: {s}"
    print(f"{s}\n   donor {meta.loc[s, 'donor']}, timepoint {meta.loc[s, 'timepoint']}, tissue {meta.loc[s, 'tissue']}")
assert meta.loc[PLACODE_STACK, "donor"] == meta.loc[RIDGE_STACK, "donor"], "different donors"
assert meta.loc[PLACODE_STACK, "timepoint"] == meta.loc[RIDGE_STACK, "timepoint"], "different timepoints"

in_p, in_r = df.stack_name == PLACODE_STACK, df.stack_name == RIDGE_STACK
print("\nniche clusters in the placode stack:", df.loc[in_p, NICHE_COL].value_counts().astype(int).to_dict())
print("niche clusters in the ridge stack:  ", df.loc[in_r, NICHE_COL].value_counts().astype(int).to_dict())
for stack_mask, ids, what in ((in_p, PLACODE_CLUSTERS, "PLACODE_CLUSTERS"),
                              (in_r, RIDGE_CLUSTERS, "RIDGE_CLUSTERS"),
                              (in_p, PLACODE_BACKGROUND_CLUSTERS or [], "PLACODE_BACKGROUND_CLUSTERS"),
                              (in_r, RIDGE_BACKGROUND_CLUSTERS or [], "RIDGE_BACKGROUND_CLUSTERS")):
    for c in ids:
        n = int((stack_mask & (df[NICHE_COL] == c)).sum())
        if n == 0:
            raise ValueError(f"{what}: cluster {c} has no cells in its stack at {NICHE_COL} "
                             f"({NICHE_CONFIG}) -- cluster numbers belong to one run and one resolution")

df["group"] = None
df.loc[in_p & df[NICHE_COL].isin(PLACODE_CLUSTERS), "group"] = "hair_placode"
df.loc[in_r & df[NICHE_COL].isin(RIDGE_CLUSTERS), "group"] = "ridge"
if PLACODE_BACKGROUND_CLUSTERS is None:
    df.loc[in_p & df["group"].isna(), "group"] = "placode_background"
    print("\nWARNING placode_background = ALL other cells in the placode stack: it may include "
          "suprabasal cells, so departures partly reflect basal vs suprabasal differences.")
else:
    df.loc[in_p & df[NICHE_COL].isin(PLACODE_BACKGROUND_CLUSTERS), "group"] = "placode_background"
if RIDGE_BACKGROUND_CLUSTERS is None:
    df.loc[in_r & df["group"].isna(), "group"] = "ridge_background"
    print("\nWARNING ridge_background = ALL other cells in the ridge stack: it may include "
          "suprabasal cells, so departures partly reflect basal vs suprabasal differences.")
else:
    df.loc[in_r & df[NICHE_COL].isin(RIDGE_BACKGROUND_CLUSTERS), "group"] = "ridge_background"

g = df[df["group"].notna()].copy()
print("\ncells per group:", g["group"].value_counts().reindex(GROUP_ORDER).to_dict())
for f in FEATURES:
    assert f in g.columns, f"feature {f} missing"

# The four groups of Fig. 7 (front-skin background, hair placode, ridge basal, ridge other).
_group_n = g["group"].value_counts().reindex(GROUP_ORDER).fillna(0).astype(int).tolist()
assert _group_n == [393, 276, 1903, 1006], (
    f"the group sizes {dict(zip(GROUP_ORDER, _group_n))} differ from the deposited run (393, 276, 1903, 1006): "
    "the niche cluster ids in the config cell belong to that run and must be re-identified "
    "if a re-run renumbers the clusters")

In [ ]:
# every cell of the two stacks; `group` is blank for front-skin cells outside the two niche clusters used
cells_out = df[(df.stack_name == PLACODE_STACK) | (df.stack_name == RIDGE_STACK)].copy()
cells_out["group_label"] = cells_out["group"].map({
    "placode_background": "Front skin (non-placodal)", "hair_placode": "Front skin (placodal)",
    "ridge": "Ridge (basal)", "ridge_background": "Ridge (non-basal)"})
cells_out["sample"] = np.where(cells_out.stack_name == PLACODE_STACK, "Front skin", "Ridge")
fig_feats = ["major_axis_z_angle", "axis_minor_length", "area", "true_sphericity"]   # panels I, J, K, L
pc_cols = [f"PC{i}" for i in range(1, N_PCS + 1)]
other = [f for f in morpho["avail_kf"] if f not in fig_feats]
cells_out = cells_out[["group", "group_label", "sample", "stack_name", "donor", "timepoint", "tissue", "pattern",
                       "label", "centroid-0", "centroid-1", "centroid-2", NICHE_COL, SHAPE_CLUSTER_COL, "qc_score"]
                      + fig_feats + pc_cols + other]
cells_out = (cells_out.rename(columns={"label": "cell_label", NICHE_COL: "niche_cluster_hop3_r1.5",
                                       SHAPE_CLUSTER_COL: "shape_cluster_leiden"})
             .sort_values(["sample", "group", "cell_label"], na_position="last").reset_index(drop=True))
print(cells_out["group"].value_counts(dropna=False).to_dict())
save_table(cells_out.set_index("cell_label"), "cells_four_groups")


## 1. Summary table — n, median, IQR per group and feature

In [ ]:
rows = []
for grp in GROUP_ORDER:
    sub = g[g["group"] == grp]
    for f in FEATURES:
        v = sub[f].dropna()
        rows.append(dict(group=grp, feature=f, n=len(v), median=v.median(),
                         q1=v.quantile(0.25), q3=v.quantile(0.75)))
summary = pd.DataFrame(rows).set_index(["feature", "group"])
display(summary.round(3).unstack("group").swaplevel(axis=1).sort_index(axis=1).reindex(columns=GROUP_ORDER, level=0))
save_table(summary.round(5), "summary_by_group")

## 2. Departure from its own background — the core result

For each feature: Cliff's δ of `hair_placode` vs `placode_background`, and of
`ridge` vs `ridge_background`, with bootstrap 95% CIs (cells resampled within
each group).  Positive δ = the structure's cells are larger on that feature
than its background's.

In [ ]:
def cliffs_delta(a, b):
    """Cliff's delta = P(a > b) - P(a < b), computed exactly via sorting."""
    a = np.asarray(a, float); b = np.sort(np.asarray(b, float))
    below = np.searchsorted(b, a, side="left")            # b < a
    above = len(b) - np.searchsorted(b, a, side="right")  # b > a
    return (below.sum() - above.sum()) / (len(a) * len(b))


def delta_with_ci(a, b, n_boot=N_BOOT, seed=SEED):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    rng = np.random.default_rng(seed)
    boots = np.array([cliffs_delta(rng.choice(a, len(a)), rng.choice(b, len(b)))
                      for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return cliffs_delta(a, b), lo, hi


def delta_table(group_a, group_b):
    A, B = g[g["group"] == group_a], g[g["group"] == group_b]
    out = {f: delta_with_ci(A[f], B[f]) for f in FEATURES}
    return pd.DataFrame(out, index=["delta", "ci_low", "ci_high"]).T


def magnitude(d):
    d = abs(d)
    return "negligible" if d < 0.147 else "small" if d < 0.33 else "medium" if d < 0.474 else "large"


dep_placode = delta_table("hair_placode", "placode_background")
dep_ridge = delta_table("ridge", "ridge_background")
departures = pd.concat({"placode_vs_its_background": dep_placode,
                        "ridge_vs_its_background": dep_ridge}, axis=1)
for key in ("placode_vs_its_background", "ridge_vs_its_background"):
    departures[(key, "size")] = departures[(key, "delta")].map(magnitude)
departures = departures.sort_index(axis=1)
display(departures.round(3))
save_table(departures.round(5), "departure_from_own_background")

r_profiles = np.corrcoef(dep_placode["delta"], dep_ridge["delta"])[0, 1]
print(f"Pearson r between the two departure profiles (across {len(FEATURES)} features): {r_profiles:.2f}")

In [ ]:
def departure_plot(dep_a, dep_b, name_a, name_b, col_a, col_b, width=760, height=560):
    """Paired lollipop: features on y, two series with bootstrap CIs, zero line."""
    feats = list(dep_a.index)[::-1]
    y = np.arange(len(feats))
    fig = go.Figure()
    for dep, name, col, dy in ((dep_a, name_a, col_a, 0.17), (dep_b, name_b, col_b, -0.17)):
        d = dep.loc[feats]
        for yi, (_, r) in zip(y + dy, d.iterrows()):          # stems
            fig.add_shape(type="line", x0=0, x1=r["delta"], y0=yi, y1=yi,
                          line=dict(color=col, width=1.5))
        fig.add_trace(go.Scatter(
            x=d["delta"], y=y + dy, mode="markers", name=name,
            marker=dict(size=10, color=col, line=dict(width=1, color="white")),
            error_x=dict(type="data", symmetric=False,
                         array=d["ci_high"] - d["delta"], arrayminus=d["delta"] - d["ci_low"],
                         color=col, thickness=1.2, width=4),
            customdata=np.stack([d["ci_low"], d["ci_high"]], axis=1),
            hovertemplate="%{y}<br>δ %{x:.2f} [%{customdata[0]:.2f}, %{customdata[1]:.2f}]<extra>" + name + "</extra>"))
    fig.add_vline(x=0, line=dict(color="#555", width=1))
    for x in (-0.474, -0.147, 0.147, 0.474):
        fig.add_vline(x=x, line=dict(color="#ccc", width=1, dash="dot"))
    fig.update_layout(**_base_layout(width, height, legend_title="<b>Departure</b>"))
    fig.update_yaxes(tickvals=list(y), ticktext=feats, showgrid=False, zeroline=False,
                     tickfont=dict(size=FONT_TICK, family=FONT_FAMILY))
    fig.update_xaxes(range=[-1.02, 1.02], zeroline=False, showgrid=False,
                     title_text="Cliff's δ (structure vs its own background)",
                     title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
                     tickfont=dict(size=FONT_TICK, family=FONT_FAMILY))
    return fig


fig = departure_plot(dep_placode, dep_ridge, "hair placode vs its background",
                     "ridge vs its background", GROUP_PALETTE["hair_placode"], GROUP_PALETTE["ridge"])
fig.show()
save(fig, "departure_lollipop")

## 3. Direct comparisons — structure vs structure, background vs background

`hair_placode` vs `ridge` next to `placode_background` vs `ridge_background`
(the tissue-background reference).  If the structure difference exceeds the
background difference, the appendages diverge beyond their tissue context.

In [ ]:
struct = delta_table("hair_placode", "ridge")
backg = delta_table("placode_background", "ridge_background")
direct = pd.concat({"placode_vs_ridge": struct, "background_vs_background": backg}, axis=1)
direct[("compare", "abs_struct_minus_abs_bg")] = struct["delta"].abs() - backg["delta"].abs()
direct[("compare", "CIs_overlap")] = ~((struct["ci_low"] > backg["ci_high"]) | (backg["ci_low"] > struct["ci_high"]))
display(direct.round(3))
save_table(direct.round(5), "direct_comparisons")

## 4. Multivariate separation in 10-D shape PC space

Euclidean distance between group medians (coordinate-wise), and the same
divided by the pooled within-group spread (mean of the two groups' median
distance to their own median).  > 1 = well separated, ≤ 1 = overlapping.

In [ ]:
pc_cols = [f"PC{i}" for i in range(1, N_PCS + 1)]
X = {grp: g.loc[g["group"] == grp, pc_cols].to_numpy(float) for grp in GROUP_ORDER}
med = {grp: np.median(X[grp], axis=0) for grp in GROUP_ORDER}
spread = {grp: float(np.median(np.linalg.norm(X[grp] - med[grp], axis=1))) for grp in GROUP_ORDER}
raw = pd.DataFrame([[np.linalg.norm(med[a] - med[b]) for b in GROUP_ORDER] for a in GROUP_ORDER],
                   index=GROUP_ORDER, columns=GROUP_ORDER)
norm = pd.DataFrame([[raw.loc[a, b] / (0.5 * (spread[a] + spread[b])) for b in GROUP_ORDER] for a in GROUP_ORDER],
                    index=GROUP_ORDER, columns=GROUP_ORDER)
print("within-group spread:", {k: round(v, 2) for k, v in spread.items()})
print("\nraw distance between medians:"); display(raw.round(3))
print("spread-normalised distance:"); display(norm.round(2))
save_table(raw.round(5), "pc_distance_raw"); save_table(norm.round(5), "pc_distance_normalised")

key = pd.Series({
    "placode vs ridge": norm.loc["hair_placode", "ridge"],
    "placode bg vs ridge bg": norm.loc["placode_background", "ridge_background"],
    "placode vs its background": norm.loc["hair_placode", "placode_background"],
    "ridge vs its background": norm.loc["ridge", "ridge_background"],
}, name="normalised distance")
print("\nKEY:"); print(key.round(2).to_string())
save_table(key.round(5).to_frame(), "pc_distance_key")

In [ ]:
fig = go.Figure(go.Heatmap(
    z=norm.values, x=GROUP_ORDER, y=GROUP_ORDER, zmin=0,
    colorscale=[[0, "#f7fbff"], [0.5, "#6baed6"], [1, "#08306b"]],
    text=norm.round(2).values, texttemplate="%{text}",
    textfont=dict(size=FONT_TICK, family=FONT_FAMILY),
    colorbar=dict(title=dict(text="distance / spread", font=dict(size=FONT_AXIS_TITLE - 1, family=FONT_FAMILY)),
                  thickness=16, len=0.78, outlinewidth=0,
                  tickfont=dict(size=FONT_TICK - 1, family=FONT_FAMILY)),
    hovertemplate="%{y} vs %{x}: %{z:.2f}<extra></extra>"))
fig.update_xaxes(type="category", tickfont=dict(size=FONT_TICK, family=FONT_FAMILY))
fig.update_yaxes(type="category", autorange="reversed", tickfont=dict(size=FONT_TICK, family=FONT_FAMILY))
fig.update_layout(plot_bgcolor="#fff", paper_bgcolor="#fff", width=640, height=520,
                  margin=dict(l=150, r=40, t=20, b=120), font=dict(family=FONT_FAMILY))
fig.show()
save(fig, "pc_distance_heatmap")

## 5a. Violins — each feature across the four groups

In [ ]:
for f in FEATURES:
    fig = violin_panel(g, f, "group", palette=GROUP_PALETTE, group_order=GROUP_ORDER,
                       y_transform="log1p" if f in LOG_FEATURES else None,
                       x_title="", include_zero=False, width=700, height=440,
                       tick_rotation=-20, random_seed=SEED)
    fig.show()
    save(fig, f"violin_{f}")

## 5b. Shape UMAP — the four groups on all cells (grey)

In [ ]:
df["group_or_none"] = df["group"]
fig = umap_highlight(df, "group_or_none", GROUP_ORDER, x_col="umap_1", y_col="umap_2",
                     palette=GROUP_PALETTE, on_top=["hair_placode", "ridge"],
                     legend_title="<b>Group</b>", seed=SEED)
fig.show()
save(fig, "shape_umap_groups")

In [ ]:
# Fig. 7G: the same four groups on the niche UMAP of the featured configuration (NICHE_CONFIG).
# The pickle holds one niche UMAP row per cell with adjacency; the other cells stay NaN and are left out.
_nr = niche_data["niche_results"][NICHE_CONFIG]
_valid = np.asarray(_nr["valid_mask"])
for _k in (1, 2):
    df[f"niche_umap_{_k}"] = np.nan
    df.loc[_valid, f"niche_umap_{_k}"] = np.asarray(_nr["umap"])[:, _k - 1]
fig = umap_highlight(df, "group_or_none", GROUP_ORDER, x_col="niche_umap_1", y_col="niche_umap_2",
                     palette=GROUP_PALETTE, on_top=["hair_placode", "ridge"],
                     legend_title="<b>Group</b>", seed=SEED)
fig.show()
save(fig, "niche_umap_groups")

## 5c. Representative cells — the 10 closest to each group's median (10-D PC space)

Rows = groups.  Needs PyVista + trimesh and the mesh files (`mesh_lookup` in
the morphospace pickle); skipped with a note if either is missing.

In [ ]:
reps = []
for gi, grp in enumerate(GROUP_ORDER):
    sub = g[g["group"] == grp]
    d = np.linalg.norm(sub[pc_cols].to_numpy(float) - med[grp], axis=1)
    for rank, idx in enumerate(sub.index[np.argsort(d)[:N_REPRESENTATIVES]]):
        key_ = (df.at[idx, "stack_name"], int(df.at[idx, "label"]))
        mp = mesh_lookup.get(key_, "")
        reps.append(dict(cluster=gi, group=grp, role=f"rep_{rank}", stack_name=key_[0],
                         label=key_[1], dist_to_median=float(d[np.argsort(d)[rank]]),
                         mesh_path=mp, mesh_exists=bool(mp) and os.path.exists(mp)))
rep_df = pd.DataFrame(reps)
display(rep_df[["group", "role", "label", "dist_to_median", "mesh_exists"]])
save_table(rep_df, "representative_cells")

In [ ]:
import matplotlib.pyplot as plt

try:
    import pyvista as pv
    import trimesh
    pv.OFF_SCREEN = True
    HAS_PYVISTA = True
except ImportError:
    HAS_PYVISTA = False

_CAM_CFGS = {
    "iso": dict(pos=(2.8, 2.2, 1.6), up=(0, 0, 1)),
    "xz":  dict(pos=(0.0, -3.2, 0.0), up=(0, 0, 1)),
    "xy":  dict(pos=(0.0, 0.0, 3.2), up=(0, 1, 0)),
    "yz":  dict(pos=(3.2, 0.0, 0.0), up=(0, 0, 1)),
}


def load_cell_mesh(mesh_path):
    """Load a cell .obj, center at origin, return (verts_xyz, faces, rms_um).

    Vertices are NOT normalized -- the caller picks the divisor so we can
    share one divisor across the whole gallery (relative sizes) or use a
    per-cell divisor (every cell fills its panel). Returns None on failure
    or for degenerate meshes.
    """
    try:
        tm = trimesh.load(mesh_path, file_type="obj", process=False)
    except Exception:
        return None
    verts = np.array(tm.vertices, dtype=float)
    faces = np.array(tm.faces, dtype=int)
    if len(verts) == 0 or len(faces) == 0:
        return None
    verts = verts[:, [2, 1, 0]]                   # ZYX -> XYZ (matches NB08 history)
    verts -= verts.mean(axis=0)
    rms = float(np.sqrt((verts ** 2).sum(axis=1).mean()))
    if rms <= 0:
        return None
    return verts, faces, rms


def render_normalized(verts_norm, faces, color="#4ECDC4",
                      camera_position="iso",
                      window_size=(800, 800),
                      background="#FFFFFF",
                      smooth_iter=30):
    """Render a pre-normalized mesh (vertices already divided by chosen divisor)."""
    pv_faces = np.hstack([np.full((len(faces), 1), 3, dtype=np.int64),
                          faces.astype(np.int64)]).ravel()
    mesh = pv.PolyData(verts_norm, pv_faces)
    if smooth_iter > 0:
        mesh = mesh.smooth(n_iter=smooth_iter, relaxation_factor=0.08)

    cfg = _CAM_CFGS.get(camera_position, _CAM_CFGS["iso"])
    pl = pv.Plotter(off_screen=True, window_size=window_size)
    pl.set_background(background)
    pl.add_mesh(mesh, color=color, smooth_shading=True,
                specular=0.45, specular_power=25,
                ambient=0.15, diffuse=0.85)
    pl.camera.position = cfg["pos"]
    pl.camera.focal_point = (0, 0, 0)
    pl.camera.up = cfg["up"]
    pl.remove_all_lights()
    pl.add_light(pv.Light(position=(3.5, 3.0, 5.0), intensity=0.80))
    pl.add_light(pv.Light(position=(-2.0, -2.0, 2.0), intensity=0.28))
    pl.add_light(pv.Light(position=(0.0, 2.0, -2.5), intensity=0.18))
    img = pl.screenshot(None, return_img=True)
    pl.close()
    return img


def render_reps_with_relative_sizes(rep_df, cmap,
                                    relative_sizes=True,
                                    camera_position="iso",
                                    window_size=(800, 800),
                                    background="#FFFFFF",
                                    smooth_iter=30,
                                    progress_every=20,
                                    progress_label=""):
    """Two-pass renderer used by every NB08 gallery cell.

    Pass 1: load all meshes referenced by ``rep_df`` (centering, RMS).
    Pass 2: render each mesh with a SHARED divisor (largest cell fills
            its panel, others render smaller) when ``relative_sizes`` is
            True, or a PER-CELL divisor (every cell fills its panel)
            when False.

    Required columns in ``rep_df``: ``cluster``, ``role`` (e.g. ``rep_0``),
    ``mesh_path``. Optional: ``mesh_exists``.

    Returns ``(thumbnails, meta)`` where ``thumbnails`` is the
    ``{(cluster_id, rep_idx): RGBA}`` dict consumed by ``build_gallery``.
    """
    # --- Pass 1: load + RMS ---
    loaded = {}
    n_target = 0
    for _, row in rep_df.iterrows():
        c = int(row["cluster"])
        role = str(row.get("role", ""))
        rep_idx = int(role.replace("rep_", "")) if role.startswith("rep_") else 0
        mp = str(row.get("mesh_path", ""))
        if not mp or not os.path.exists(mp):
            continue
        n_target += 1
        built = load_cell_mesh(mp)
        if built is None:
            continue
        loaded[(c, rep_idx)] = built

    if not loaded:
        return {}, {"mode": "none", "n_loaded": 0, "n_target": n_target,
                    "mode_desc": "no meshes loaded"}

    rms_values = [d[2] for d in loaded.values()]
    global_rms = max(rms_values)
    smallest_rms = min(rms_values)

    if relative_sizes:
        shared_divisor = global_rms * 1.8
        def divisor(_rms):
            return shared_divisor
        mode_desc = (f"relative sizes (shared divisor; largest RMS "
                     f"= {global_rms:.2f} um, smallest = {smallest_rms:.2f} um)")
    else:
        def divisor(rms):
            return rms * 1.8
        mode_desc = "per-cell (every cell rescaled to fill its panel)"

    # --- Pass 2: render with chosen divisor ---
    thumbnails = {}
    n_done = 0
    for (c, rep_idx), (verts, faces, rms) in loaded.items():
        color = cmap.get(c, "#888888")
        try:
            img = render_normalized(
                verts / divisor(rms), faces, color=color,
                camera_position=camera_position,
                window_size=window_size, background=background,
                smooth_iter=smooth_iter,
            )
        except Exception as e:
            print(f"  {progress_label}c={c} rep_{rep_idx}: render error: {e}")
            continue
        if img is not None:
            thumbnails[(c, rep_idx)] = img
        n_done += 1
        if progress_every and n_done % progress_every == 0:
            print(f"  Rendered {n_done} / {len(loaded)} ...")

    meta = {
        "mode": "relative" if relative_sizes else "per_cell",
        "mode_desc": mode_desc,
        "n_loaded": len(loaded),
        "n_target": n_target,
        "global_rms_um": global_rms,
        "smallest_rms_um": smallest_rms,
    }
    return thumbnails, meta


def build_gallery(thumbnails, cluster_ids, cmap,
                  n_cols_per_cluster=None,
                  orientation="vertical",
                  cell_size=2.5, save_path=None, dpi=600,
                  cluster_label_prefix="N",
                  suptitle=None):
    """
    thumbnails: {(cluster_id, rep_index): RGBA}
    orientation: 'vertical' (rows=clusters) or 'horizontal' (cols=clusters)
    suptitle:    optional figure title (e.g. relative-size mode description).
    """
    if n_cols_per_cluster is None:
        n_cols_per_cluster = max((idx for (_, idx) in thumbnails.keys()),
                                 default=0) + 1

    n_clusters = len(cluster_ids)
    if orientation == "vertical":
        n_rows, n_cols = n_clusters, n_cols_per_cluster
    else:
        n_rows, n_cols = n_cols_per_cluster, n_clusters

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(cell_size * n_cols, cell_size * n_rows),
                             facecolor="white")
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    if n_cols == 1:
        axes = axes.reshape(-1, 1)

    for ax in axes.flat:
        ax.set_facecolor("white")
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_linestyle(":"); sp.set_linewidth(0.7); sp.set_color("#BBBBBB")

    for ci, c in enumerate(cluster_ids):
        color_hex = cmap.get(c, "#888888")
        for ri in range(n_cols_per_cluster):
            if orientation == "vertical":
                row_idx, col_idx = ci, ri
            else:
                row_idx, col_idx = ri, ci
            ax = axes[row_idx, col_idx]
            key = (c, ri)
            if key in thumbnails:
                ax.imshow(thumbnails[key])
            else:
                ax.text(0.5, 0.5, "--", ha="center", va="center",
                        fontsize=9, color="#999", transform=ax.transAxes)
            if orientation == "vertical" and ri == 0:
                ax.set_ylabel(f"{cluster_label_prefix}{c}",
                              fontsize=10, color=color_hex,
                              fontweight="bold", labelpad=4,
                              fontfamily=FONT_FAMILY)
            elif orientation == "horizontal" and ri == 0:
                ax.set_title(f"{cluster_label_prefix}{c}",
                             fontsize=10, color=color_hex,
                             fontweight="bold", pad=4,
                             fontfamily=FONT_FAMILY)

    if suptitle:
        fig.suptitle(suptitle, y=1.0, fontsize=10, color="#444",
                     family=FONT_FAMILY)
    plt.tight_layout(pad=0.3)
    if save_path:
        # TIFF gets LZW compression (Illustrator + Photoshop read it transparently)
        sp = str(save_path)
        kw = dict(dpi=dpi, bbox_inches="tight", facecolor="white")
        if sp.lower().endswith((".tif", ".tiff")):
            kw["pil_kwargs"] = {"compression": "tiff_lzw"}
        plt.savefig(sp, **kw)
        print(f"Saved: {sp}")
    plt.show()
    return fig


if not HAS_PYVISTA:
    print("[skipped] PyVista / trimesh not installed -- representative meshes not rendered")
elif not rep_df["mesh_exists"].any():
    print("[skipped] none of the mesh files were found (mesh_lookup paths point elsewhere)")
else:
    group_cmap = {gi: GROUP_PALETTE[grp] for gi, grp in enumerate(GROUP_ORDER)}
    thumbs, meta_ = render_reps_with_relative_sizes(rep_df, group_cmap, relative_sizes=True)
    print(meta_["mode_desc"])
    gal = build_gallery(thumbs, list(range(len(GROUP_ORDER))), group_cmap,
                        n_cols_per_cluster=N_REPRESENTATIVES, orientation="vertical",
                        cluster_label_prefix="", save_path=None)
    for gi, grp in enumerate(GROUP_ORDER):     # row labels = group names
        gal.axes[gi * N_REPRESENTATIVES].set_ylabel(grp, fontsize=9, color=GROUP_PALETTE[grp],
                                                     fontweight="bold")
    os.makedirs(FIG_OUT_DIR, exist_ok=True)
    # out = os.path.join(FIG_OUT_DIR, FIG_PREFIX + "representative_cells.png")
    out = os.path.join(FIG_OUT_DIR, FIG_PREFIX + "representative_cells.tif")
    gal.savefig(out, dpi=600, bbox_inches="tight", facecolor="white")
    print("Saved:", out)

## Plain-language summary

In [ ]:
def departs(dep):
    """Features where the CI excludes 0 and |delta| is at least small."""
    hit = dep[((dep["ci_low"] > 0) | (dep["ci_high"] < 0)) & (dep["delta"].abs() >= 0.147)]
    return [f"{f} ({'higher' if r['delta'] > 0 else 'lower'}, {magnitude(r['delta'])}, δ={r['delta']:.2f})"
            for f, r in hit.sort_values("delta", key=abs, ascending=False).iterrows()]


print("SUMMARY (donor2, 13 weeks; descriptive, one donor)\n")
print(f"Cells: " + ", ".join(f"{k} {v:,}" for k, v in g['group'].value_counts().reindex(GROUP_ORDER).items()))
print("\nHair placode departs from its own background on:")
for s in departs(dep_placode) or ["no feature beyond a negligible shift"]:
    print("   -", s)
print("\nRidge departs from its own background on:")
for s in departs(dep_ridge) or ["no feature beyond a negligible shift"]:
    print("   -", s)
same_sign = int((np.sign(dep_placode["delta"]) == np.sign(dep_ridge["delta"])).sum())
print(f"\nDeparture profiles: Pearson r = {r_profiles:.2f}; same direction on {same_sign}/{len(FEATURES)} features "
      f"-> " + ("they largely AGREE (a shared way of departing from the surrounding epidermis)" if r_profiles > 0.5
               else "they DIVERGE (each structure departs in its own way)" if r_profiles < 0
               else "they agree only partly"))
pr, bb = key["placode vs ridge"], key["placode bg vs ridge bg"]
print(f"\n10-D shape space: normalised d(placode, ridge) = {pr:.2f} vs d(placode bg, ridge bg) = {bb:.2f} -> " +
      ("the structures are FURTHER apart than their backgrounds" if pr > bb * 1.2
       else "the structures are CLOSER together than their backgrounds" if pr < bb * 0.8
       else "the structures are about as far apart as their backgrounds (within +/-20%)"))
print(f"Each structure vs its own background: placode {key['placode vs its background']:.2f}, "
      f"ridge {key['ridge vs its background']:.2f}")
print("\nReminder: structures were picked by niche clusters built from the same shape PCs, so some "
      "separation from their backgrounds is expected by construction; read the comparisons "
      "relative to the backgrounds.")

## 6. Four-group statistics for the Fig. 7 violins

This section tests the four groups against each other.  Per feature: **Welch's one-way ANOVA** (the
groups differ in size 276-1,903 and in spread, so the unequal-variance form is used; the plain F
is also written to the CSV) with **Games-Howell** post hoc for the six pairs, and as a
distribution-free check **Kruskal-Wallis** with **Dunn's test, Holm-adjusted**.  Effect sizes:
Hedges' g per pair, omega-squared / epsilon-squared per feature.  Size is tested on log1p(volume),
as plotted.  Stars: * p < 0.05, ** < 0.01, *** < 0.001, **** < 0.0001.

Caveat (same as the policy above): cells are the replicates and this is one donor, so the
p-values describe this donor.  `scipy`'s studentized-range tail bottoms out near 1e-13, so smaller
Games-Howell p-values are reported as "< 1e-12".  `STAT_FEATURES` can be any features in `g`.

In [ ]:
from scipy import stats
import itertools as _it
_st = stats

STAT_FEATURES = {                 # column -> (panel, transform); any feature of `g` can be added
    "major_axis_z_angle": ("I", None),
    "axis_minor_length":  ("J", None),
    "area":               ("K", "log1p"),
    "true_sphericity":    ("L", None),
}
STAT_PAIRS = list(_it.combinations(GROUP_ORDER, 2))
PAIRS = STAT_PAIRS

def welch_anova(samples):
    """Welch (1951) one-way ANOVA. Returns F, df1, df2, p."""
    k = len(samples)
    n = np.array([len(s) for s in samples], float)
    m = np.array([s.mean() for s in samples])
    v = np.array([s.var(ddof=1) for s in samples])
    w = n / v
    mw = (w * m).sum() / w.sum()
    num = ((w * (m - mw) ** 2).sum()) / (k - 1)
    lam = (3 * ((1 - w / w.sum()) ** 2 / (n - 1)).sum()) / (k * k - 1)
    F = num / (1 + 2 * (k - 2) * lam / 3)
    df1, df2 = k - 1, 1 / lam
    return F, df1, df2, stats.f.sf(F, df1, df2)


def omega_squared(samples):
    allv = np.concatenate(samples); N = len(allv); k = len(samples)
    ss_b = sum(len(s) * (s.mean() - allv.mean()) ** 2 for s in samples)
    ss_w = sum(((s - s.mean()) ** 2).sum() for s in samples)
    ms_w = ss_w / (N - k)
    return (ss_b - (k - 1) * ms_w) / (ss_b + ss_w + ms_w)


def games_howell(a, b, k):
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    se2 = va / na + vb / nb
    t = (a.mean() - b.mean()) / np.sqrt(se2)
    df = se2 ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    p = stats.studentized_range.sf(abs(t) * np.sqrt(2), k, df)
    return t, df, float(min(p, 1.0))


def dunn(ranks, groups):
    """Dunn z for every pair on the pooled ranks (tie-corrected); returns {pair: (z, p_raw)}."""
    N = len(ranks)
    _, cnt = np.unique(ranks, return_counts=True)
    tie = 1 - (cnt ** 3 - cnt).sum() / (N ** 3 - N)
    out = {}
    for ga, gb in PAIRS:
        ra, rb = ranks[groups == ga], ranks[groups == gb]
        se = np.sqrt((N * (N + 1) / 12) * tie * (1 / len(ra) + 1 / len(rb)))
        z = (ra.mean() - rb.mean()) / se
        out[(ga, gb)] = (z, 2 * stats.norm.sf(abs(z)))
    return out


def holm(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(running, 1.0)
    return adj


def hedges_g(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    g = (a.mean() - b.mean()) / sp
    return g * (1 - 3 / (4 * (na + nb) - 9))


def stars(p):
    return "****" if p < 1e-4 else "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "ns"


def four_group_stats(g, features=STAT_FEATURES):
    desc, omni, pairs = [], [], []
    for col, (panel, tf) in features.items():
        vals_all = np.log1p(g[col]) if tf == "log1p" else g[col]
        samples = [vals_all[g.group == grp].dropna().to_numpy(float) for grp in GROUP_ORDER]
        for grp, s in zip(GROUP_ORDER, samples):
            q1, q3 = np.percentile(s, [25, 75])
            desc.append(dict(panel=panel, feature=col, transform=tf or "none", group=grp, n=len(s),
                             mean=s.mean(), sd=s.std(ddof=1), median=np.median(s), q1=q1, q3=q3))
        F, d1, d2, pW = welch_anova(samples); Fc, pc = _st.f_oneway(*samples); Hh, pH = _st.kruskal(*samples)
        N = sum(len(s) for s in samples); k = len(samples)
        omni.append(dict(panel=panel, feature=col, transform=tf or "none", n_total=N, k=k,
                         welch_F=F, welch_df1=d1, welch_df2=d2, welch_p=pW, welch_stars=stars(pW),
                         anova_F=Fc, anova_df1=k - 1, anova_df2=N - k, anova_p=pc, omega_squared=omega_squared(samples),
                         kruskal_H=Hh, kruskal_df=k - 1, kruskal_p=pH, kruskal_stars=stars(pH),
                         epsilon_squared=(Hh - k + 1) / (N - k)))
        grps = g["group"].to_numpy(); dz = dunn(_st.rankdata(vals_all.to_numpy(float)), grps)
        rows = []
        for ga, gb in STAT_PAIRS:
            a = vals_all[g.group == ga].to_numpy(float); b = vals_all[g.group == gb].to_numpy(float)
            t, df_, pgh = games_howell(a, b, k); z, pd_raw = dz[(ga, gb)]
            rows.append(dict(panel=panel, feature=col, transform=tf or "none", group_a=ga, group_b=gb,
                             n_a=len(a), n_b=len(b), mean_a=a.mean(), mean_b=b.mean(),
                             median_a=np.median(a), median_b=np.median(b),
                             mean_diff_a_minus_b=a.mean() - b.mean(), hedges_g=hedges_g(a, b),
                             games_howell_t=t, games_howell_df=df_, games_howell_p=pgh, dunn_z=z, dunn_p_raw=pd_raw))
        pr = pd.DataFrame(rows)
        pr["dunn_p_holm"] = holm(pr["dunn_p_raw"])
        pr["games_howell_p_report"] = pr["games_howell_p"].map(lambda p: "<1e-12" if p < 1e-12 else f"{p:.3g}")
        pr["games_howell_stars"] = pr["games_howell_p"].map(stars)
        pr["dunn_holm_stars"] = pr["dunn_p_holm"].map(stars)
        pr["tests_agree"] = pr["games_howell_stars"].eq("ns") == pr["dunn_holm_stars"].eq("ns")
        pairs.append(pr)
    return pd.DataFrame(desc), pd.DataFrame(omni), pd.concat(pairs, ignore_index=True)


stat_desc, stat_omni, stat_pairs = four_group_stats(g)
display(stat_omni[["panel", "feature", "welch_F", "welch_df2", "welch_p", "omega_squared", "kruskal_H", "kruskal_p", "epsilon_squared"]].round(3))
display(stat_pairs[["panel", "group_a", "group_b", "hedges_g", "games_howell_p_report", "games_howell_stars", "dunn_p_holm", "dunn_holm_stars", "tests_agree"]].round(4))
save_table(stat_desc.round(5).set_index(["feature", "group"]), "fourgroup_descriptives")
save_table(stat_omni.set_index("feature"), "fourgroup_omnibus_tests")
save_table(stat_pairs.set_index(["feature", "group_a", "group_b"]), "fourgroup_pairwise_posthoc")


### 6b. The Fig. 7 I-L violins with significance bars

Same `violin_panel` as the figure.  Bars = Games-Howell stars (`star_col="dunn_holm_stars"` for the
rank-based test); `FOUR_COMPARISONS` are the four the text makes, pass `STAT_PAIRS` for all six.
Display rules: densities clipped to the data range (no mass beyond 90 deg / the sphericity
bound), the y axis stops at the data or physical bound and the bars sit above it, n under every group
(dots = up to 500 cells per group; boxes and densities use all cells), size ticked in raw voxels on the log1p scale, bounded
features shown over their bound (tilt 0-90 deg, sphericity 0.5-1).

In [ ]:
STYLE = dict(point_opacity=0.75, point_size=3.0, max_points=500,   # up to 500 cells per group as dots, 75 % opaque
             violin_line_width=2.0, show_grid=False, axis_lines=True,
             bar_level=0.055, bar_font=12, show_n=True, width=560, height=460)
# y axis per feature: (range or None = data range with 3 % padding, tick values or None, title)
AXES = {
    "major_axis_z_angle": ((0, 90), list(range(0, 91, 15)), "Long-axis angle to z (°)"),
    "axis_minor_length":  (None, None, "Cell width, minor axis (µm)"),
    "area":               (None, [50, 100, 200, 500, 1000, 2000], "Cell volume (µm³, log scale)"),
    "true_sphericity":    ((0.5, 1.0), [0.5, 0.6, 0.7, 0.8, 0.9, 1.0], "True sphericity"),
}
FOUR_COMPARISONS = [("placode_background", "hair_placode"), ("ridge", "ridge_background"),
                    ("hair_placode", "ridge"), ("placode_background", "ridge_background")]
VIOLIN_X_LABELS = {"placode_background": "Front-skin epidermis", "hair_placode": "Hair placode",
                   "ridge": "Ridge (basal)", "ridge_background": "Ridge sample, other cells"}


def pack_levels(comparisons, pos):
    """Assign each comparison a level so bars on one level never overlap. Short bars first."""
    levels, placed = [], []
    for ga, gb in sorted(comparisons, key=lambda p: abs(pos[p[0]] - pos[p[1]])):
        x0, x1 = sorted((pos[ga], pos[gb]))
        for li, lv in enumerate(levels):
            if all(x1 < a or x0 > b for a, b in lv):
                lv.append((x0, x1)); placed.append((ga, gb, x0, x1, li)); break
        else:
            levels.append([(x0, x1)]); placed.append((ga, gb, x0, x1, len(levels) - 1))
    return placed, len(levels)


def add_bars_above(fig, pairs_df, feature, comparisons, group_order, star_col="games_howell_stars",
                   level=STYLE["bar_level"], font_family=FONT_FAMILY):
    """Bars in paper coordinates above the plot area; shrinks the y-axis domain to make room."""
    pos = {g: i for i, g in enumerate(group_order)}
    rows = pairs_df[pairs_df.feature == feature].set_index(["group_a", "group_b"])
    placed, n_levels = pack_levels(comparisons, pos)
    top = 1 - (n_levels + 0.3) * level            # axis domain ends here
    fig.update_yaxes(domain=[0, top])
    for ga, gb, x0, x1, li in placed:
        key = (ga, gb) if (ga, gb) in rows.index else (gb, ga)
        txt = rows.loc[key, star_col]
        y = top + level * (li + 0.45); tick = 0.3 * level
        fig.add_shape(type="path", path=f"M {x0} {y - tick} L {x0} {y} L {x1} {y} L {x1} {y - tick}",
                      line=dict(color="#333", width=1), xref="x", yref="paper")
        fig.add_annotation(x=(x0 + x1) / 2, y=y, text=txt, showarrow=False, xref="x", yref="paper",
                           yanchor="bottom", yshift=-1 if txt == "ns" else -5,
                           font=dict(size=STYLE["bar_font"] - 3 if txt == "ns" else STYLE["bar_font"], color="#333", family=font_family))


def style_violin(fig, col, g, group_order, labels, axes=AXES, style=STYLE, font_family=FONT_FAMILY):
    """The figure style and display rules: clipped densities, bounded axes, raw-unit ticks, n labels."""
    rng_, ticks, title = axes.get(col, (None, None, col))
    tf = STAT_FEATURES[col][1] if col in STAT_FEATURES else None
    vals = np.log1p(g[col]) if tf == "log1p" else g[col]
    fig.update_traces(selector=dict(type="violin"), spanmode="hard", line_width=style["violin_line_width"])
    if rng_ is None:
        lo, hi = float(vals.min()), float(vals.max()); pad = 0.03 * (hi - lo); rng_ = (lo - pad, hi + pad)
    fig.update_yaxes(range=list(rng_), title_text=title)
    if ticks is not None:
        tv = [np.log1p(t) for t in ticks] if tf == "log1p" else ticks
        fig.update_yaxes(tickvals=tv, ticktext=[f"{t:,}" if isinstance(t, int) else str(t) for t in ticks])
    n = g["group"].value_counts()
    text = [labels[x] + (f"<br>n = {n[x]:,}" if style["show_n"] else "") for x in group_order]
    fig.update_xaxes(title_text="", ticktext=text, tickangle=0, showgrid=style["show_grid"])
    fig.update_yaxes(showgrid=style["show_grid"])
    if style["axis_lines"]:   # open frame: left + bottom lines, outward ticks
        fig.update_yaxes(showline=True, linecolor="#333", linewidth=1, ticks="outside", tickcolor="#333", ticklen=4)
        fig.update_xaxes(showline=True, linecolor="#333", linewidth=1, ticks="outside", tickcolor="#333", ticklen=4)
    fig.update_layout(margin=dict(b=70 if style["show_n"] else 50, t=10))


def violin_with_bars(col, comparisons=FOUR_COMPARISONS, star_col="games_howell_stars"):
    """Densities clipped to the data, axis stops at the data / physical bound, bars above the axis,
    n under each group, size ticked in raw µm³."""
    tf = STAT_FEATURES[col][1]
    fig = violin_panel(g, col, "group", palette=GROUP_PALETTE, group_order=GROUP_ORDER,
                       y_transform=tf, x_title="", include_zero=False, width=STYLE["width"], height=STYLE["height"],
                       show_n_annotation=False, random_seed=0, point_opacity=STYLE["point_opacity"],
                       point_size=STYLE["point_size"], max_points=STYLE["max_points"])
    style_violin(fig, col, g, GROUP_ORDER, VIOLIN_X_LABELS)
    add_bars_above(fig, stat_pairs, col, comparisons, GROUP_ORDER, star_col=star_col)
    return fig


for col, (panel, _) in STAT_FEATURES.items():
    fig = violin_with_bars(col)
    fig.show()
    save(fig, f"violin_{col}_stats_Fig7{panel}")


### 6c. Key for the significance bars (separate small figure)

In [ ]:
def star_key(test_line="Games-Howell post hoc after Welch's ANOVA", width=300, height=150):
    """Standalone key for the significance bars (a separate figure file, not drawn on the panels)."""
    rows = [("****", "p < 0.0001"), ("***", "p < 0.001"), ("**", "p < 0.01"), ("*", "p < 0.05"), ("ns", "not significant")]
    fig = go.Figure()
    fig.add_annotation(x=0, y=1, xref="paper", yref="paper", xanchor="left", yanchor="top", showarrow=False,
                       text=test_line, font=dict(size=11, color="#333", family=FONT_FAMILY))
    for i, (sym, txt) in enumerate(rows):
        y = 0.78 - i * 0.17
        fig.add_annotation(x=0.02, y=y, xref="paper", yref="paper", xanchor="left", yanchor="middle", showarrow=False,
                           text=sym, font=dict(size=10 if sym == "ns" else 14, color="#333", family=FONT_FAMILY))
        fig.add_annotation(x=0.28, y=y, xref="paper", yref="paper", xanchor="left", yanchor="middle", showarrow=False,
                           text=txt, font=dict(size=11, color="#333", family=FONT_FAMILY))
    fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
    fig.update_layout(width=width, height=height, plot_bgcolor="#fff", paper_bgcolor="#fff",
                      margin=dict(l=10, r=10, t=10, b=10), showlegend=False)
    return fig


fig = star_key()
fig.show()
save(fig, "significance_key")      # separate file; place it once beside the Fig. 7 I-L panels
